In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

def apply_multi_model_business_logic(df, variance_threshold=1.5, models=['LGBM', 'IsolationForest', 'Survival']):
    """
    Applies corporate risk rules to the Machine Learning confidence intervals for MULTIPLE models
    to generate final, firm quotes for comparison.

    Parameters
    ----------
    df : pandas.DataFrame
        The prediction dataset output from Phase 3 containing P50 and P90 columns for each model.
    variance_threshold : float, optional (default=1.5)
        The maximum acceptable difference in days between P50 and P90 before padding the quote.
    models : list of str
        The prefix names of the models evaluated.

    Returns
    -------
    df : pandas.DataFrame
        The dataframe with new AI quoted transit days for each model.
    """
    print(f"Applying business logic (Risk Variance Threshold: {variance_threshold} days)...")

    for model in models:
        p50_col = f'Pred_{model}_P50_Days'
        p90_col = f'Pred_{model}_P90_Days'
        quoted_col = f'{model}_Quoted_Transit_Days'

        if p50_col in df.columns and p90_col in df.columns:
            # Calculate the variance spread for this specific model
            risk_spread = df[p90_col] - df[p50_col]

            # Apply logic: Use P90 if spread > threshold, else use P50
            df[quoted_col] = np.where(
                risk_spread > variance_threshold,
                np.ceil(df[p90_col]),
                np.ceil(df[p50_col])
            )
    return df


def compare_model_performance(df, actual_col, models=['LGBM', 'IsolationForest', 'Survival']):
    """
    Simulates network performance for all models and outputs a comparative leaderboard
    to analyze which algorithm best protects the NSL while keeping quotes competitive.

    Returns
    -------
    best_model : str
        The name of the best performing model to use for the final reports.
    """
    print("\n" + "="*70)
    print(" === CAPSTONE MODEL COMPARISON & NSL EVALUATION ===")
    print("="*70)

    # 1. Calculate Historical Baseline
    if 'Quoted_Transit_Days' in df.columns:
        df['Historical_Delivered_in_Commit'] = (df[actual_col] <= df['Quoted_Transit_Days']).astype(int)
        historical_nsl = df['Historical_Delivered_in_Commit'].mean() * 100
        avg_hist_quote = df['Quoted_Transit_Days'].mean()
    else:
        historical_nsl = 0
        avg_hist_quote = 0

    print(f"{'Model Type':<20} | {'Simulated NSL %':<15} | {'Avg Quoted Days':<15} | {'NSL Improvement'}")
    print("-" * 70)
    print(f"{'Historical (Static)':<20} | {historical_nsl:<15.2f} | {avg_hist_quote:<15.2f} | Baseline")

    best_model = None
    highest_nsl = -1

    # 2. Evaluate Each AI Model
    for model in models:
        quoted_col = f'{model}_Quoted_Transit_Days'
        if quoted_col in df.columns:
            df[f'{model}_Delivered_in_Commit'] = (df[actual_col] <= df[quoted_col]).astype(int)
            model_nsl = df[f'{model}_Delivered_in_Commit'].mean() * 100
            avg_quote = df[quoted_col].mean()
            improvement = model_nsl - historical_nsl

            # Identify the best model
            if model_nsl > highest_nsl:
                highest_nsl = model_nsl
                best_model = model

            print(f"{model + ' (AI)':<20} | {model_nsl:<15.2f} | {avg_quote:<15.2f} | +{improvement:.2f}%")

    print("="*70)
    print(f"🏆 Recommendation: Deploy {best_model} model for maximum SLA protection.\n")
    return best_model


def generate_postal_adjustment_report(df, best_model):
    """Groups the simulation results by destination postal code based on the WINNING model."""
    print(f"Generating Postal Code Adjustment Report using {best_model}...")

    quoted_col = f'{best_model}_Quoted_Transit_Days'

    if 'Quoted_Transit_Days' not in df.columns or 'dest_pstl_cd' not in df.columns:
        print("[WARNING] Missing required columns to generate postal adjustment report.")
        return None

    group_cols = ['dest_pstl_cd']
    if 'Pickup_Day_of_Week' in df.columns:
        day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
        df['Pickup_Day_Name'] = df['Pickup_Day_of_Week'].map(day_map).fillna(df['Pickup_Day_of_Week'])
        group_cols.append('Pickup_Day_Name')

    # Calculate days adjusted
    df['Days_Adjusted'] = df[quoted_col] - df['Quoted_Transit_Days']

    postal_summary = df.groupby(group_cols).agg(
        Total_Shipments=(quoted_col, 'count'),
        Historical_Avg_Quote=('Quoted_Transit_Days', 'mean'),
        AI_Avg_Quote=(quoted_col, 'mean'),
        Avg_Days_Added=('Days_Adjusted', 'mean')
    ).reset_index()

    postal_summary['Historical_Avg_Quote'] = postal_summary['Historical_Avg_Quote'].round(1)
    postal_summary['AI_Avg_Quote'] = postal_summary['AI_Avg_Quote'].round(1)
    postal_summary['Avg_Days_Added'] = postal_summary['Avg_Days_Added'].round(1)

    postal_summary = postal_summary.sort_values(by=['Avg_Days_Added', 'Total_Shipments'], ascending=[False, False])

    print("\n=== TOP 10 POSTAL CODES REQUIRING TRANSIT TIME INCREASES (BY DAY) ===")
    print(postal_summary.head(10).to_string(index=False))
    print("=======================================================================\n")
    return postal_summary


def forecast_manual_buffer_impact(df, postal_summary, actual_col, top_n=5, buffer_options=[1, 2]):
    """Simulates manual buffer additions based on the bottlenecks identified."""
    if postal_summary is None or 'Quoted_Transit_Days' not in df.columns: return

    print(f"Forecasting NSL impact of manually buffering the Top {top_n} worst postal codes...")
    worst_postals = postal_summary['dest_pstl_cd'].unique()[:top_n]
    historical_nsl = (df[actual_col] <= df['Quoted_Transit_Days']).mean() * 100
    print(f"Baseline Historical NSL: {historical_nsl:.2f}%")

    for buffer in buffer_options:
        hypothetical_quotes = df['Quoted_Transit_Days'].copy()
        mask = df['dest_pstl_cd'].isin(worst_postals)
        hypothetical_quotes.loc[mask] += buffer

        hypothetical_nsl = (df[actual_col] <= hypothetical_quotes).mean() * 100
        improvement = hypothetical_nsl - historical_nsl
        print(f" - If +{buffer} Day(s) added to Top {top_n}: NSL jumps to {hypothetical_nsl:.2f}% (Improvement: +{improvement:.2f}%)")
    print("=======================================================================\n")


# --- Execution Block ---
if __name__ == "__main__":
    # Pointing to the NEW Advanced Phase 3 output
    input_file = "Phase3_Advanced_Predictions.csv"
    input_path = os.path.join("..", "Data", "Processed", input_file)
    output_path = os.path.join("..", "Data", "Processed", "Phase4_Final_Simulation.csv")
    postal_report_path = os.path.join("..", "Data", "Processed", "Phase4_Postal_Adjustments.csv")
    original_data_path = os.path.join("..", "Data", "Processed", "Cleaned_IEF_Shipments_FY26.csv")

    try:
        print(f"Loading multi-model predictions from {input_path}...")
        results_df = pd.read_csv(input_path)

        # --- MOCK DATA GENERATOR (For immediate testing) ---
        # If you haven't run the new Phase 3 yet, this creates dummy predictions so the script doesn't crash
        # and you can immediately see how the comparison leaderboard looks.
        if 'Pred_IsolationForest_P50_Days' not in results_df.columns and 'Pred_LGBM_P50_Days' not in results_df.columns:
             # Fallback if only the old basic predictions exist
            if 'Pred_P50_Days' in results_df.columns:
                print("[INFO] IsolationForest/Survival data missing. Simulating model variance for evaluation display...")
                results_df['Pred_LGBM_P50_Days'] = results_df['Pred_P50_Days']
                results_df['Pred_LGBM_P90_Days'] = results_df['Pred_P90_Days']

                # Mocking Isolation Forest: Random Forest base (similar to P50) with wider variance simulating the +4 anomaly buffer
                results_df['Pred_IsolationForest_P50_Days'] = results_df['Pred_P50_Days'] * 0.95
                results_df['Pred_IsolationForest_P90_Days'] = results_df['Pred_P90_Days'] * 1.15

                results_df['Pred_Survival_P50_Days'] = results_df['Pred_P50_Days'] * 1.02
                results_df['Pred_Survival_P90_Days'] = results_df['Pred_P90_Days'] * 0.98
        # ---------------------------------------------------

        if os.path.exists(original_data_path):
            original_df = pd.read_csv(original_data_path)
            le = LabelEncoder()
            le.fit(original_df['dest_pstl_cd'].astype(str))

            # Identify columns that contain postal codes and decode if present
            if 'dest_pstl_cd' in results_df.columns:
                # Ensure we only decode if they are integers/floats from the label encoding
                if pd.api.types.is_numeric_dtype(results_df['dest_pstl_cd']):
                     results_df['dest_pstl_cd'] = le.inverse_transform(results_df['dest_pstl_cd'].astype(int))

        target_col = 'Calculated_Actual_Days' if 'Calculated_Actual_Days' in results_df.columns else 'Actual_Transit_Days'
        model_list = ['LGBM', 'IsolationForest', 'Survival']

        # 1. Apply Logic for ALL models
        applied_logic_df = apply_multi_model_business_logic(results_df, variance_threshold=1.5, models=model_list)

        # 2. Compare Models and output leaderboard
        winning_model = compare_model_performance(applied_logic_df, target_col, models=model_list)

        # 3. Generate Reports based on the winner
        postal_adjustments_df = generate_postal_adjustment_report(applied_logic_df, best_model=winning_model)
        forecast_manual_buffer_impact(applied_logic_df, postal_adjustments_df, target_col, top_n=5, buffer_options=[1, 2])

        # 4. Save final output
        applied_logic_df.to_csv(output_path, index=False)
        print(f"Final simulated dataset saved to {output_path} successfully.")

        if postal_adjustments_df is not None:
            postal_adjustments_df.to_csv(postal_report_path, index=False)
            print(f"Postal code adjustment report saved to {postal_report_path} successfully.")

    except FileNotFoundError:
        print(f"[ERROR] Could not find {input_path}.")
        print("Ensure you have run 'phase3_advanced_modeling.py' to generate the predictions first.")

Loading multi-model predictions from ..\Data\Processed\Phase3_Advanced_Predictions.csv...
Applying business logic (Risk Variance Threshold: 1.5 days)...

 === CAPSTONE MODEL COMPARISON & NSL EVALUATION ===
Model Type           | Simulated NSL % | Avg Quoted Days | NSL Improvement
----------------------------------------------------------------------
Historical (Static)  | 68.77           | 7.41            | Baseline
LGBM (AI)            | 87.44           | 10.40           | +18.67%
IsolationForest (AI) | 64.52           | 8.02            | +-4.26%
Survival (AI)        | 63.21           | 7.63            | +-5.56%
🏆 Recommendation: Deploy LGBM model for maximum SLA protection.

Generating Postal Code Adjustment Report using LGBM...

=== TOP 10 POSTAL CODES REQUIRING TRANSIT TIME INCREASES (BY DAY) ===
dest_pstl_cd Pickup_Day_Name  Total_Shipments  Historical_Avg_Quote  AI_Avg_Quote  Avg_Days_Added
        3019       Wednesday                1                   5.0          11.0         